# CARLA Python API - Lab3


Lab goals:
- reuse helper functions for faster experiments;
- follow vehicles with the spectator camera;
- inspect waypoints and road topology with debug drawing;
- compare repositioning (`set_transform`) with real driving control (`apply_control`).

## CARLA documentation links

- Main docs index: https://carla.readthedocs.io/en/latest/
- Python API reference: https://carla.readthedocs.io/en/latest/python_api/
- Core actors and world API: https://carla.readthedocs.io/en/latest/core_actors/
- Map and waypoints API: https://carla.readthedocs.io/en/latest/core_map/
- Debug helper (drawing points/text): https://carla.readthedocs.io/en/latest/python_api/#carladebughelper
- Traffic Manager (autopilot behavior): https://carla.readthedocs.io/en/latest/adv_traffic_manager/

## Optional troubleshooting: Conda in VS Code notebooks

If your notebook kernel cannot import `carla`, use this quick check.

### Quick check

In a VS Code Python notebook, click `Select Kernel` (top-right), then run:

```python
import carla
```

Install `ipykernel` if VS Code asks for it.

![Python notebook kernel selection](img/python-kernel.png)

### Conda works in terminal but not in VS Code

1. Find the absolute path of `conda.exe` in terminal (example: `C:\anaconda3\Scripts\conda.exe`).
2. Open VS Code settings (`Ctrl + ,`).
3. Search for `python.condaPath`.
4. Paste the path in `Python: Conda Path`.
   ![Python conda path](img/python-conda-path.png)
5. Restart VS Code and reselect the kernel.

In [1]:
import carla, time, pygame, random

pygame 2.6.1 (SDL 2.28.4, Python 3.7.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
client = carla.Client('localhost', 2000)
client.set_timeout(10.0)

world = client.get_world()
spectator = world.get_spectator()

## 1. Helper functions used in this notebook

This lab reuses three utility functions:
- `move_spectator_to(...)`: place the camera behind/above a target transform.
- `spawn_vehicle(...)`: spawn a vehicle from the blueprint library at a chosen spawn point.
- `draw_on_screen(...)`: convenience wrapper around `world.debug.draw_string(...)`.

Keep these helpers stable and focus your edits on behavior logic in later sections.

In [3]:
def move_spectator_to(transform, spectator, distance=7.0, x=0, y=0, z=3.0, yaw=0, pitch=-15, roll=0):
    back_location = transform.location - transform.get_forward_vector() * distance
    
    back_location.x += x
    back_location.y += y
    back_location.z += z
    transform.rotation.yaw += yaw
    transform.rotation.pitch = pitch
    transform.rotation.roll = roll
    
    spectator_transform = carla.Transform(back_location, transform.rotation)
    
    spectator.set_transform(spectator_transform)

def spawn_vehicle(world, vehicle_index=0, spawn_index=0):
    blueprint_library = world.get_blueprint_library()
    vehicle_bp = blueprint_library.filter('vehicle.*')[vehicle_index]
    spawn_point = world.get_map().get_spawn_points()[spawn_index]
    vehicle = world.spawn_actor(vehicle_bp, spawn_point)
    return vehicle

def draw_on_screen(world, transform, content="O", color=carla.Color(0, 255, 0), life_time=20):
    world.debug.draw_string(transform.location, content, color=color, life_time=life_time)


## 2. Position the spectator near an autonomous vehicle

In this section you spawn a vehicle, enable autopilot, and place the spectator near it once.

### Procedure
1. Spawn a vehicle with `spawn_vehicle(world)`.
2. Enable autopilot with `vehicle.set_autopilot(True)`.
3. Call `move_spectator_to(vehicle.get_transform(), spectator)` once.
4. Advance the simulation with `world.tick()` while the vehicle moves.

Stop with `KeyboardInterrupt`, then destroy the actor with `vehicle.destroy()`.

In [ ]:
vehicle = spawn_vehicle(world)
vehicle.set_autopilot(True)
move_spectator_to(vehicle.get_transform(), spectator)

while True:
    world.tick()

KeyboardInterrupt: 

: 

In [ ]:
vehicle.destroy()

True

: 

## 3. Waypoints

Waypoints represent navigable points on the road network and are useful for local path inspection.

In the next code cell:
1. a vehicle is spawned and moved by autopilot;
2. at fixed intervals, the closest waypoint is sampled;
3. sampled waypoint locations are printed for inspection.

![waypoints](img/waypoints.png)

### Example of waypoint sampling

Waypoints can be computed from actor position with `world.get_map().get_waypoint(location)`.

In [ ]:
vehicle = spawn_vehicle(world)
vehicle.set_autopilot(True)

waypoints_list = []
transform = vehicle.get_transform()
move_spectator_to(transform, spectator)
for _ in range(10):
    transform = vehicle.get_transform()
    waypoint = world.get_map().get_waypoint(transform.location)
    waypoints_list.append(waypoint)
    world.tick()
    time.sleep(1)

for wp in waypoints_list:
    print(wp.transform.location)

KeyboardInterrupt: 

In [ ]:
vehicle.destroy()

NameError: name 'vehicle' is not defined

### Debug utilities: draw elements on screen

Use `world.debug` to visualize geometry while developing logic.

Main methods:
- `world.debug.draw_arrow`
- `world.debug.draw_box`
- `world.debug.draw_line`
- `world.debug.draw_point`
- `world.debug.draw_string`

Guidelines:
- use short `life_time` values during iteration, so the scene stays readable;
- keep color usage consistent per concept (for example, one color for current waypoint, one for target).

In [ ]:
for wp in waypoints_list:
    world.debug.draw_string(
        wp.transform.location,
        'Waypoint Here!',
        color=carla.Color(r=0, g=255, b=0),
        life_time=20.0, # life_time=0 means infinite
    )

NameError: name 'waypoints_list' is not defined

### Roads

`world.get_map().get_topology()` returns road segments as waypoint pairs `(start_wp, end_wp)`.

The next cells show how to:
1. pick an initial waypoint;
2. traverse forward with `waypoint.next(distance)`;
3. draw visited points to inspect the road structure.

In [ ]:
roads = world.get_map().get_topology()

waypoint = roads[0][0]
draw_on_screen(world, waypoint.transform)

In [ ]:
for i in range(1000):
    # get the next waypoint with distance 1
    waypoint = waypoint.next(1)[0]
    draw_on_screen(world, waypoint.transform)
    time.sleep(0.01)

KeyboardInterrupt: 

Now we draw multiple roads with different colors to make segment boundaries easier to read.

In [ ]:
def color_generator():
    colors = [
        carla.Color(r=r, g=g, b=b) for r in [0, 255] for g in [0, 255] for b in [0, 255]
    ]
    while True:
        for color in colors:
            yield color

for i, color in zip(range(len(roads)), color_generator()):
    waypoint = roads[i][0]
    for j in range(1000):
        waypoint = waypoint.next(3)[0]
        draw_on_screen(world, waypoint.transform, color=color, life_time=10)
    time.sleep(0.5)

KeyboardInterrupt: 

## 4. Repositioning vs controlling a vehicle

Two operations can look similar on screen but are fundamentally different.

- **Repositioning (`set_transform`)**: instant teleport to a new pose.
- **Controlling (`apply_control`)**: dynamic motion through throttle, steering, and brake.

Use repositioning for setup/debug. Use control commands for actual driving behavior.

In [ ]:
vehicle = spawn_vehicle(world)
roads = world.get_map().get_topology()
waypoint = roads[0][0]

vehicle.set_transform(waypoint.transform)
move_spectator_to(waypoint.transform, spectator)

A sequence of `set_transform` calls can create a movement-like visual effect, but it is still teleportation.
No physics-based driving logic is applied.

In [ ]:
sleep_time = 1
waypoint_distance = 1

try:
    while True:
        waypoint = waypoint.next(waypoint_distance)[0]
        vehicle.set_transform(waypoint.transform)
        time.sleep(sleep_time)
except KeyboardInterrupt:
    pass
finally:
    vehicle.destroy()

### Control a vehicle

In this section the vehicle is moved through `carla.VehicleControl`, which is the correct interface for driving.

Suggested workflow:
1. create small helpers (for example `move_forward`, `steer_left`);
2. apply one command at a time;
3. tick the world and sleep briefly to observe each action.

In [ ]:
vehicle = spawn_vehicle(world)
move_spectator_to(vehicle.get_transform(), spectator)

throttle = 0.6

def move_forward(vehicle, duration):
    control = carla.VehicleControl()
    control.throttle = throttle
    vehicle.apply_control(control)
    start_time = time.time()
    while time.time() - start_time < duration:
        world.tick()
        time.sleep(0.1)

def steer_left(vehicle):
    control = carla.VehicleControl()
    control.throttle = throttle
    control.steer = -0.38
    vehicle.apply_control(control)
    world.tick()
    time.sleep(2)

try:
    move_forward(vehicle, 5)
    steer_left(vehicle)
    move_forward(vehicle, 20)
finally:
    vehicle.destroy()

KeyboardInterrupt: 

### Example of random movement

This demo applies random throttle and steering values in a loop.
It is useful for testing control responsiveness, but it is not a stable navigation policy.

In [ ]:
vehicle = spawn_vehicle(world)
move_spectator_to(vehicle.get_transform(), spectator)

def random_control(vehicle):
    control = carla.VehicleControl()
    control.throttle = random.uniform(0.5, 1.0)
    control.steer = random.uniform(-1.0, 1.0)
    vehicle.apply_control(control)

try:
    while True:
        random_control(vehicle)
        world.tick()
        time.sleep(0.1)
except KeyboardInterrupt:
    pass
finally:
    vehicle.destroy()

## 5. Exercises (in class)

Use only functions and patterns already introduced in this notebook.

### Exercise 1: Multi-view spectator

Prepare three possible camera views and choose one before the vehicle starts moving:
- `1`: chase view (behind the car)
- `2`: top view (high above the car)
- `3`: side view (right side of the car)

Implementation hints:
1. Keep one variable `camera_mode`.
2. Set `camera_mode` before the main loop starts.
3. Call `move_spectator_to(...)` once before the loop, using the selected mode.
4. Keep autopilot enabled for the vehicle.

Success criteria:
- the chosen initial view is applied before the loop starts;
- no camera jumps while the vehicle is moving;
- clean shutdown with `vehicle.destroy()` and `pygame.quit()`.

In [ ]:
# TEMPLATE - Exercise 1: Multi-view spectator
# STEP 1: spawn a vehicle and enable autopilot
vehicle = None  # TODO: replace with spawn_vehicle(world)
# TODO: vehicle.set_autopilot(True)
vehicle = spawn_vehicle(world)
vehicle.set_autopilot(True)
move_spectator_to(vehicle.get_transform(), spectator)

# STEP 2: initialize pygame window
pygame.init()
pygame.display.set_mode((400, 200))
camera_mode = 3

# STEP 3: choose the initial view before entering the loop
if camera_mode == 1: 
    move_spectator_to(vehicle.get_transform(), spectator, distance=7.0, z=3.0, pitch=-15)
elif camera_mode == 2: 
    move_spectator_to(vehicle.get_transform(), spectator, distance=0.0, z=35.0, pitch=-90)
elif camera_mode == 3: 
    move_spectator_to(vehicle.get_transform(), spectator, distance=0.0, y=-3.0, z=3.0, yaw=90, pitch=-15)


running = True

try:
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

        world.tick()
        time.sleep(0.03)
finally:
    # STEP 6: cleanup
    vehicle.destroy()
    pygame.quit()
    pass

NameError: name 'spawn_vehicle' is not defined

### Exercise 2: Safety ring around the car

Goal: draw a moving ring of debug markers around the vehicle.

Simplified approach:
1. At each loop, read `center = vehicle.get_transform().location`.
2. Create a fixed list of offsets (front, back, left, right, diagonals).
3. For each offset, draw one marker (`draw_point` or `draw_string`) at `center + offset`.
4. Use short `life_time` and redraw every tick.

Success criteria:
- markers stay centered on the vehicle while it moves;
- old markers fade quickly and the screen stays readable;
- ring size is easy to adjust by changing one variable.

In [6]:
# TEMPLATE - Exercise 2: Safety ring around the car
# STEP 1: spawn a vehicle and enable autopilot
vehicle = spawn_vehicle(world)
vehicle.set_autopilot(True)
move_spectator_to(vehicle.get_transform(), spectator)

ring_radius = 3.0
ring_height = 0.6

# STEP 2: define offsets around the car center
# Add at least 8 points: front/back/left/right + diagonals
offsets = [
    carla.Location(ring_radius, 0.0, ring_height),
    carla.Location(-ring_radius, 0.0, ring_height),
    carla.Location(0.0, ring_radius, ring_height),
    carla.Location(0.0, -ring_radius, ring_height),
]

# Optional: place the spectator near the vehicle once before the loop starts
move_spectator_to(vehicle.get_transform(), spectator)

try:
    while True:
        # STEP 3: get current center from vehicle transform
        center = vehicle.get_transform().location

        # STEP 4: draw all markers around the center
        for offset in offsets:
            world.debug.draw_point(center + offset,
                                    0.05,
                                    color=carla.Color(r=26, g=63, b=10),
                                    life_time=20.0)
            pass
        world.tick()
        time.sleep(0.03)
except KeyboardInterrupt:
    pass
finally:
    # STEP 5: cleanup
    vehicle.destroy()
    pass

: 

### Exercise 3: Smooth manual driving

Build a basic manual controller with gradual commands:
- `w`: increase throttle step by step
- `s`: apply brake
- `a` / `d`: steer left/right
- no steering key: slowly return steer to `0`

Implementation hints:
1. Keep a `carla.VehicleControl()` object and update it every frame.
2. Clamp throttle and steer values to valid ranges.
3. Use a small update step (for example `0.02` to `0.05`) for smoother behavior.

Success criteria:
- vehicle starts and stops smoothly;
- steering is progressive, not abrupt;
- releasing keys leaves the car in a stable state.

In [ ]:
# TEMPLATE - Exercise 3: Smooth manual driving
# STEP 1: spawn vehicle + init pygame window
vehicle = spawn_vehicle(world)
move_spectator_to(vehicle.get_transform(), spectator)

# STEP 2: initialize pygame window
pygame.init()
pygame.display.set_mode((400, 200))

control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=0.0)
throttle_step = 0.03
steer_step = 0.04
running = True


def clamp(value, low, high):
    return max(low, min(high, value))


# STEP 6: optional one-time spectator placement before entering the loop
move_spectator_to(vehicle.get_transform(), spectator)

try:
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False


        keys = pygame.key.get_pressed()

        # STEP 3: throttle/brake logic (w/s)
        if keys == pygame.K_w:
            control.throttle = clamp(control.throttle + throttle_step, 0.0, 1.0)
            control.brake = 0.0
        elif keys == pygame.K_s:
            control.throttle = clamp(control.throttle - throttle_step, 0.0, 1.0)

        # STEP 4: steering logic (a/d) + return to zero
        if keys == pygame.K_a:
            control.steer = clamp(control.steer - steer_step, -1.0, 1.0)
        elif keys == pygame.K_d:
            control.steer = clamp(control.steer + steer_step, -1.0, 1.0)
            

        if keys == pygame.KMOD_NONE:
            if control.steer > 0:


        # STEP 5: clamp values and apply control
        control.throttle = clamp(control.throttle, 0.0, 1.0)
        control.brake = clamp(control.brake, 0.0, 1.0)
        control.steer = clamp(control.steer, -1.0, 1.0)
        vehicle.apply_control(control)
        world.tick()
        time.sleep(0.03)
finally:
    # STEP 7: cleanup
    vehicle.destroy()
    pygame.quit()
    pass

## 6. Optional home exercises

### Optional Exercise 1: Simple cruise control

Add a cruise mode toggled by `c`:
- when enabled, keep speed in a target range (for example 20-25 km/h);
- when disabled, return to manual controls.

Implementation hints:
1. Read vehicle speed from velocity magnitude.
2. If speed is below target, increase throttle slightly.
3. If speed is above target, reduce throttle or apply light brake.
4. Disable cruise automatically when `w` or `s` is pressed.

Success criteria:
- cruise keeps speed near target on straight roads;
- manual override works immediately;
- mode status is printed on screen or console.

In [ ]:
# TEMPLATE - Optional Exercise 1: Simple cruise control
# STEP 1: spawn vehicle + pygame init
vehicle = None  # TODO

control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=0.0)
cruise_enabled = False
target_min_kmh = 20.0
target_max_kmh = 25.0
running = True


# Optional: place the spectator near the vehicle once before the loop starts
# TODO: move_spectator_to(vehicle.get_transform(), spectator)

def speed_kmh(actor):
    v = actor.get_velocity()
    return 3.6 * ((v.x * v.x + v.y * v.y + v.z * v.z) ** 0.5)


try:
    while running:
        # STEP 2: events
        # - ESC -> quit
        # - C -> toggle cruise_enabled
        # TODO

        keys = pygame.key.get_pressed()
        # spd = speed_kmh(vehicle)

        # STEP 3: manual override
        # if w or s is pressed -> cruise_enabled = False
        # TODO

        # STEP 4: when cruise is ON, keep speed in [target_min_kmh, target_max_kmh]
        # TODO

        # STEP 5: when cruise is OFF, use manual throttle/brake
        # TODO
        # STEP 6: apply control + tick
        # TODO
        world.tick()
        time.sleep(0.04)
finally:
    # STEP 7: cleanup
    # TODO
    pass

### Optional Exercise 2: Short waypoint route

Create a short route of 20-30 waypoints ahead of the car and try to follow it.

Implementation hints:
1. Get current waypoint from vehicle location.
2. Build a list with repeated `next(distance)` calls.
3. Draw route waypoints with debug markers.
4. Move the car with simple steering toward the next target waypoint.

Success criteria:
- the route is visible and ordered;
- the car reaches most waypoints without teleporting;
- end condition stops the loop and cleans up actors.

In [ ]:
# TEMPLATE - Optional Exercise 2: Short waypoint route
# STEP 1: spawn vehicle
vehicle = None  # TODO

# STEP 2: create route starting from current waypoint
route = []
# TODO:
# - get current waypoint from vehicle location
# - append next waypoints (20-30 total)

# STEP 3: draw route for debug (optional)
# TODO: world.debug.draw_string / draw_point

control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=0.0)
target_index = 0

# Optional: place the spectator near the vehicle once before the loop starts
# TODO: move_spectator_to(vehicle.get_transform(), spectator)

try:
    while target_index < len(route):
        # STEP 4: select current target waypoint
        # target_wp = route[target_index]

        # STEP 5: compute simple steering toward target
        # TODO

        # STEP 6: set throttle/brake and apply control
        # TODO
        # STEP 7: switch to next waypoint when close enough
        # TODO

        world.tick()
        time.sleep(0.05)

    # STEP 8: stop vehicle at the end
    # TODO
except KeyboardInterrupt:
    pass
finally:
    # STEP 9: cleanup
    # TODO
    pass